# Helpers to work with files of data

## Setup
imdb data set as an example

In [ ]:
from pathlib import Path
import tensorflow as tf

root = "https://ai.stanford.edu/~amaas/data/sentiment/"
filename = "aclImdb_v1.tar.gz"
filepath = tf.keras.utils.get_file(filename, root + filename, extract=True,
                                   cache_dir=".")
if "_extracted" in filepath:
    path = Path(filepath) / "aclImdb"
else:
    path = Path(filepath).with_name("aclImdb")


In [ ]:
# define paths
train_pos_dr = path / "train" / "pos"
train_neg_dr = path / "train" / "neg"
test_pos_dr = path / "test" / "pos"
test_neg_dr = path / "test" / "neg"

## Parse files with Dataset

List files as Path objects:

In [ ]:
from pathlib import Path
import numpy as np

def list_txt(dirpath: Path) -> list[Path]:
    return list(dirpath.glob("*.txt"))

# list files as Path objects
train_pos_files = list_txt(train_pos_dr)
train_neg_files = list_txt(train_neg_dr)
test_pos_files  = list_txt(test_pos_dr)
test_neg_files  = list_txt(test_neg_dr)


In [ ]:
from numpy.random import shuffle
from tensorflow.data import Dataset
from typing import Sequence

def make_text_ds(filepaths: Sequence[Path]) -> Dataset:
  paths_as_str = [str(p) for p in filepaths]
  ds = Dataset.from_tensor_slices(paths_as_str)
  ds = ds.map(tf.io.read_file, num_parallel_calls = tf.data.AUTOTUNE)
  return ds

def make_labeled_ds(
    pos_files: Sequence[str],
    neg_files: Sequence[str],
    shuffle=True,
    cache=False):

  pos_ds = make_text_ds(pos_files).map(lambda x: (x, 1))
  neg_ds = make_text_ds(neg_files).map(lambda x: (x, 0))
  ds = pos_ds.concatenate(neg_ds)

  if shuffle:
    ds = ds.shuffle(len(ds))

  if cache:
    ds = ds.cache()

  return ds

train_ds = make_labeled_ds(train_pos_files, train_neg_files, shuffle=True)
test_ds = make_labeled_ds(test_pos_files, test_neg_files, shuffle=False)

In [ ]:
for X, y in train_ds.take(3):
  print(X)
  print(y)
  print()

tf.Tensor(b'This is the last Dutch language film Paul Verhoeven made before going on to make mainstream Hollywood films "Basic Instinct," "Robocop," and "Total Recall," among others. He sets the stage by opening this story with a black widow spider catching prey in her web before we meet Gerard Reve, an annoying self-centered writer with a morbid imagination. Gerard has been invited to be the guest speaker at a Literary Club meeting in sea-side town an hour or so from Amsterdam. Verhoeven lets us have glimpses of how Gerard\'s imagination twists reality. Asked if writers are a bit close to insanity he admits when he reads the newspaper "and it says \'boom\' I read \'doom,\' when it says \'flood\' I read \'blood,\' when it says \'red\' I see \'dead.\'" When he tells a story enough times he begins to believe it; "I lie the truth." He accepts an offer to be the overnight guest of the Club treasurer, a beautiful wealthy salon owner. As he gets to know her and learns her husband has died, h

In [ ]:
def prefetch(ds, batch_size = 32):
  return ds.batch(batch_size).prefetch(1)

train_set = prefetch(train_ds)
test_set = prefetch(test_ds)

## Big files

check Apache Beam